# 11 — AIA ResNet18 Physics-Safe Multi-Fold Full-Natural Benchmark

Runs all usable chronological full-natural ResNet18 image-only folds in one notebook.

Default folds:
- test_2013: train 2010–2011, validation 2012, test 2013
- test_2014: train 2010–2012, validation 2013, test 2014
- test_2015: train 2010–2013, validation 2014, test 2015

By default, completed fold metrics are reused to save GPU cost. Set `RERUN_EXISTING_FOLDS = True` in the code cell only if a fresh rerun is required.

Physics-safe rules: manifest label `label_48h_final` only; no embedded NPZ labels; no random flips/rotations/crops; deterministic resize only; train-only robust channel statistics per fold; validation-selected max-TSS threshold applied unchanged to test.


In [ ]:

from pathlib import Path
import json, hashlib, random, subprocess, time
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = str

def repo_root():
    try:
        return Path(subprocess.check_output(['git','rev-parse','--show-toplevel'], text=True).strip())
    except Exception:
        return Path.cwd()

ROOT = repo_root()
METRICS_DIR = ROOT / 'results' / 'metrics'
MODELS_DIR = ROOT / 'results' / 'models'
FIG_DIR = ROOT / 'results' / 'figures'
for d in [METRICS_DIR, MODELS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Repo root:', ROOT)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

@dataclass
class Config:
    label_col: str = 'label_48h_final'
    cache_dir: str = 'cache/gcs_npz_alexnet_fold2015'
    image_size: int = 224
    in_channels: int = 6
    batch_size: int = 32
    num_workers: int = 2
    epochs: int = 6
    learning_rate: float = 1e-4
    weight_decay: float = 5e-4
    dropout: float = 0.30
    stats_max_images: int = 2000
    stats_pixels_per_channel_per_image: int = 512
    threshold_grid_step: float = 0.0025
    seed: int = 42

CFG = Config()
CACHE = ROOT / CFG.cache_dir
CACHE.mkdir(parents=True, exist_ok=True)

FOLDS = [
    {'fold_id': 'test_2013', 'train_years': [2010, 2011], 'val_years': [2012], 'test_years': [2013]},
    {'fold_id': 'test_2014', 'train_years': [2010, 2011, 2012], 'val_years': [2013], 'test_years': [2014]},
    {'fold_id': 'test_2015', 'train_years': [2010, 2011, 2012, 2013], 'val_years': [2014], 'test_years': [2015]},
]
RERUN_EXISTING_FOLDS = False
print(CFG)
print('Folds:', FOLDS)
print('RERUN_EXISTING_FOLDS:', RERUN_EXISTING_FOLDS)

def standardise_manifest_columns(df, source_name):
    df = df.copy()
    if 'gcp_path' not in df.columns:
        for c in ['gcs_path', 'npz_path', 'file_path', 'path']:
            if c in df.columns:
                df['gcp_path'] = df[c]
                break
    if 'gcp_path' not in df.columns:
        raise ValueError(f'{source_name} has no gcp_path-like column')
    if CFG.label_col not in df.columns:
        raise ValueError(f'{source_name} has no {CFG.label_col}')
    df[CFG.label_col] = df[CFG.label_col].astype(int)
    if 'year' not in df.columns:
        if 'timestamp' in df.columns:
            df['year'] = pd.to_datetime(df['timestamp']).dt.year
        elif 'date' in df.columns:
            df['year'] = pd.to_datetime(df['date']).dt.year
        else:
            extracted = df['gcp_path'].astype(str).str.extract(r'/(20\d{2})/|_(20\d{2})\d{4}_')
            df['year'] = pd.to_numeric(extracted.bfill(axis=1).iloc[:, 0], errors='coerce').astype('Int64')
    df['year'] = df['year'].astype(int)
    return df

def load_sample_universe():
    triplets = [
        (
            ROOT / 'results/metrics/aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_train_samples.csv',
            ROOT / 'results/metrics/aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_val_samples.csv',
            ROOT / 'results/metrics/aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_test_samples.csv',
        ),
        (
            ROOT / 'local_archive/aborted_06_full_attempt/aia_alexnet_fold2015_benchmark_full_train_samples.csv',
            ROOT / 'local_archive/aborted_06_full_attempt/aia_alexnet_fold2015_benchmark_full_val_samples.csv',
            ROOT / 'local_archive/aborted_06_full_attempt/aia_alexnet_fold2015_benchmark_full_test_samples.csv',
        ),
    ]
    frames, used = [], []
    for triplet in triplets:
        if all(p.exists() for p in triplet):
            for p in triplet:
                frames.append(standardise_manifest_columns(pd.read_csv(p), str(p)))
                used.append(str(p.relative_to(ROOT)))
            break
    if not frames:
        raise FileNotFoundError('Full-natural sample CSV triplet not found. Restore sample CSVs from Notebook 09 or local_archive.')
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['gcp_path']).reset_index(drop=True)
    print('Loaded sample files:')
    for u in used:
        print(' -', u)
    return df

universe = load_sample_universe()
year_summary = universe.groupby('year')[CFG.label_col].agg(rows='count', positives='sum').reset_index()
year_summary['negatives'] = year_summary['rows'] - year_summary['positives']
year_summary['positive_rate'] = year_summary['positives'] / year_summary['rows']
display(year_summary)
print('Universe rows:', len(universe))
print('Universe years:', sorted(universe['year'].unique().tolist()))

def local_path_for_gcs(gcp_path):
    return CACHE / (hashlib.md5(gcp_path.encode('utf-8')).hexdigest() + '_' + Path(gcp_path).name)

def build_fold_splits(fold):
    tr = universe[universe['year'].isin(fold['train_years'])].copy(); tr['split'] = 'train'
    va = universe[universe['year'].isin(fold['val_years'])].copy(); va['split'] = 'val'
    te = universe[universe['year'].isin(fold['test_years'])].copy(); te['split'] = 'test'
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

all_required = set()
fold_summaries = []
for fold in FOLDS:
    tr, va, te = build_fold_splits(fold)
    for name, df in [('train', tr), ('val', va), ('test', te)]:
        rows = len(df); pos = int(df[CFG.label_col].sum()); neg = rows - pos
        fold_summaries.append({'fold_id': fold['fold_id'], 'split': name, 'years': str(sorted(df['year'].unique().tolist())), 'rows': rows, 'positives': pos, 'negatives': neg, 'positive_rate': pos / rows if rows else np.nan})
        assert rows > 0 and pos > 0, (fold['fold_id'], name, rows, pos)
        all_required.update(df['gcp_path'].astype(str).tolist())
missing = [p for p in sorted(all_required) if not local_path_for_gcs(p).exists()]
missing_path = METRICS_DIR / 'aia_resnet18_physics_safe_multifold_fullnatural_missing_gcs_paths.txt'
missing_path.write_text('\n'.join(missing) + ('\n' if missing else ''))
display(pd.DataFrame(fold_summaries))
print('required_unique:', len(all_required))
print('cached:', len(all_required) - len(missing))
print('missing:', len(missing))
print('missing_list:', missing_path.relative_to(ROOT))
print('first_missing:', missing[:5])
if missing:
    raise RuntimeError(f'Cache incomplete: missing {len(missing)} files. Prefetch before training.')

def pick_npz_array(npz):
    for k in ['x', 'X', 'image', 'images', 'data', 'arr_0']:
        if k in npz.files:
            arr = npz[k]
            if isinstance(arr, np.ndarray) and arr.ndim >= 2:
                return arr
    for k in npz.files:
        arr = npz[k]
        if isinstance(arr, np.ndarray) and arr.ndim >= 2:
            return arr
    raise ValueError(f'No image-like array found in NPZ keys={npz.files}')

def to_chw_six(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = np.squeeze(arr)
    if arr.ndim != 3:
        raise ValueError(f'Expected 3D array, got {arr.shape}')
    if arr.shape[0] == CFG.in_channels:
        return arr.astype(np.float32)
    if arr.shape[-1] == CFG.in_channels:
        return np.transpose(arr, (2, 0, 1)).astype(np.float32)
    raise ValueError(f'Cannot infer six-channel layout from {arr.shape}')

def load_raw_chw(gcp_path):
    with np.load(local_path_for_gcs(gcp_path), allow_pickle=False) as npz:
        arr = pick_npz_array(npz)
    return np.nan_to_num(to_chw_six(arr), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def deterministic_pixel_sample(channel_2d, n, seed_key):
    flat = channel_2d.reshape(-1)
    if flat.size <= n:
        return flat.astype(np.float32)
    h = int(hashlib.md5(seed_key.encode('utf-8')).hexdigest()[:8], 16)
    rng = np.random.default_rng(h)
    idx = rng.choice(flat.size, size=n, replace=False)
    return flat[idx].astype(np.float32)

def compute_train_channel_stats(train_df, fold_prefix, progress_log):
    stats_path = METRICS_DIR / f'{fold_prefix}_train_channel_stats.json'
    if stats_path.exists():
        with open(progress_log, 'a') as fh:
            fh.write(f'Loading existing channel stats: {stats_path}\n')
        return json.loads(stats_path.read_text())
    with open(progress_log, 'a') as fh:
        fh.write('Computing train-derived robust channel statistics...\n')
    train_paths = train_df['gcp_path'].drop_duplicates().tolist()
    if CFG.stats_max_images and len(train_paths) > CFG.stats_max_images:
        train_paths = train_paths[:CFG.stats_max_images]
    samples_by_channel = [[] for _ in range(CFG.in_channels)]
    t0 = time.time()
    for i, gcp_path in enumerate(train_paths, 1):
        chw = load_raw_chw(gcp_path)
        for c in range(CFG.in_channels):
            samples_by_channel[c].append(deterministic_pixel_sample(chw[c], CFG.stats_pixels_per_channel_per_image, f'{fold_prefix}|{gcp_path}|{c}'))
        if i % 100 == 0 or i == len(train_paths):
            msg = f'stats progress: {i}/{len(train_paths)} images elapsed_min={(time.time() - t0) / 60:.1f}'
            print(msg, flush=True)
            with open(progress_log, 'a') as fh:
                fh.write(msg + '\n')
    channel_stats = []
    for c in range(CFG.in_channels):
        vals = np.concatenate(samples_by_channel[c]).astype(np.float32)
        vals = vals[np.isfinite(vals)]
        q01, q25, q50, q75, q99 = np.percentile(vals, [1, 25, 50, 75, 99])
        scale = float(q75 - q25)
        if not np.isfinite(scale) or scale <= 1e-6:
            scale = float(np.std(vals) + 1e-6)
        channel_stats.append({'channel_index': c, 'q01': float(q01), 'q25': float(q25), 'median': float(q50), 'q75': float(q75), 'q99': float(q99), 'iqr_or_std': float(scale), 'sample_count': int(vals.size)})
    payload = {'normalisation': 'train_derived_channel_robust_q01_q99_clip_median_iqr_scale', 'source_split': 'train_only', 'fold_prefix': fold_prefix, 'num_train_paths_used': len(train_paths), 'channel_stats': channel_stats}
    stats_path.write_text(json.dumps(payload, indent=2))
    return payload

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.dropout = nn.Dropout2d(dropout) if dropout and dropout > 0 else nn.Identity()
        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(nn.Conv2d(in_planes, planes, 1, stride, bias=False), nn.BatchNorm2d(planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)

class ResNet18AIA(nn.Module):
    def __init__(self, in_channels=6, dropout=0.30):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(nn.Conv2d(in_channels, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        self.layer1 = self._make_layer(64, 2, 1, dropout / 2)
        self.layer2 = self._make_layer(128, 2, 2, dropout / 2)
        self.layer3 = self._make_layer(256, 2, 2, dropout / 2)
        self.layer4 = self._make_layer(512, 2, 2, dropout / 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, 1)
    def _make_layer(self, planes, blocks, stride, dropout):
        layers = [BasicBlock(self.in_planes, planes, stride, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.in_planes, planes, 1, dropout))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = self.avgpool(x); x = torch.flatten(x, 1); x = self.dropout(x)
        return self.fc(x).squeeze(1)

def safe_auc(y_true, y_prob, kind='roc'):
    y_true = np.asarray(y_true); y_prob = np.asarray(y_prob)
    if len(np.unique(y_true)) < 2:
        return float('nan')
    return float(roc_auc_score(y_true, y_prob)) if kind == 'roc' else float(average_precision_score(y_true, y_prob))

def confusion_at_threshold(y_true, y_prob, thr):
    y_true = np.asarray(y_true).astype(int); y_pred = (np.asarray(y_prob) >= thr).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum()); tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum()); fn = int(((y_true == 1) & (y_pred == 0)).sum())
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    tss = recall + specificity - 1.0
    total = tp + tn + fp + fn
    pe = ((tp + fp) * (tp + fn) + (fn + tn) * (fp + tn)) / (total * total) if total else 0.0
    hss = (accuracy - pe) / (1 - pe) if abs(1 - pe) > 1e-12 else 0.0
    return {'threshold': float(thr), 'accuracy': float(accuracy), 'precision': float(precision), 'recall': float(recall), 'specificity': float(specificity), 'f1': float(f1), 'tss': float(tss), 'hss': float(hss), 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

def threshold_grid_metrics(y_true, y_prob):
    thresholds = np.arange(0.0, 1.0 + CFG.threshold_grid_step, CFG.threshold_grid_step)
    grid = pd.DataFrame([confusion_at_threshold(y_true, y_prob, t) for t in thresholds])
    best = grid.sort_values(['tss', 'hss', 'threshold'], ascending=[False, False, True]).iloc[0].to_dict()
    return grid, best

class AIANPZPhysicsSafeDataset(Dataset):
    def __init__(self, df, channel_stats_payload):
        self.df = df.reset_index(drop=True)
        stats = channel_stats_payload['channel_stats']
        self.clip_lo = np.array([s['q01'] for s in stats], dtype=np.float32)[:, None, None]
        self.clip_hi = np.array([s['q99'] for s in stats], dtype=np.float32)[:, None, None]
        self.median = np.array([s['median'] for s in stats], dtype=np.float32)[:, None, None]
        scale = np.array([s['iqr_or_std'] for s in stats], dtype=np.float32)[:, None, None]
        self.scale = np.where(np.abs(scale) < 1e-6, 1.0, scale).astype(np.float32)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; gcp_path = row['gcp_path']; y = np.float32(row[CFG.label_col])
        x = load_raw_chw(gcp_path); x = np.clip(x, self.clip_lo, self.clip_hi); x = (x - self.median) / self.scale
        x = torch.from_numpy(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32))
        if x.shape[-2:] != (CFG.image_size, CFG.image_size):
            x = F.interpolate(x.unsqueeze(0), size=(CFG.image_size, CFG.image_size), mode='bilinear', align_corners=False).squeeze(0)
        return x, torch.tensor(y, dtype=torch.float32), gcp_path

def make_loaders(train_df, val_df, test_df, stats):
    common = dict(num_workers=CFG.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=CFG.num_workers > 0)
    return (
        DataLoader(AIANPZPhysicsSafeDataset(train_df, stats), batch_size=CFG.batch_size, shuffle=True, **common),
        DataLoader(AIANPZPhysicsSafeDataset(val_df, stats), batch_size=CFG.batch_size, shuffle=False, **common),
        DataLoader(AIANPZPhysicsSafeDataset(test_df, stats), batch_size=CFG.batch_size, shuffle=False, **common),
    )

@torch.no_grad()
def predict(model, loader, split_name, progress_log):
    model.eval(); y_true = []; y_prob = []; paths = []; t0 = time.time()
    for batch_idx, (xb, yb, batch_paths) in enumerate(loader, 1):
        xb = xb.to(device, non_blocking=True); logits = model(xb); prob = torch.sigmoid(logits).detach().cpu().numpy()
        y_prob.extend(prob.tolist()); y_true.extend(yb.numpy().astype(int).tolist()); paths.extend(list(batch_paths))
        if batch_idx % 100 == 0 or batch_idx == len(loader):
            msg = f'predict split={split_name} batch={batch_idx}/{len(loader)} elapsed_min={(time.time() - t0) / 60:.1f}'
            print(msg, flush=True)
            with open(progress_log, 'a') as fh:
                fh.write(msg + '\n')
    return pd.DataFrame({'gcp_path': paths, 'y_true': y_true, 'y_prob': y_prob})

def evaluate_split(model, loader, split_name, progress_log):
    pred = predict(model, loader, split_name, progress_log)
    y_true = pred['y_true'].values; y_prob = pred['y_prob'].values
    grid, best = threshold_grid_metrics(y_true, y_prob)
    metrics = {'roc_auc': safe_auc(y_true, y_prob, 'roc'), 'pr_auc': safe_auc(y_true, y_prob, 'pr'), 'brier_score': float(brier_score_loss(y_true, y_prob)), 'at_0_5': confusion_at_threshold(y_true, y_prob, 0.5), 'best_tss': best}
    return metrics, grid, pred

def summarise_split(df, name):
    pos = int(df[CFG.label_col].sum()); rows = int(len(df))
    return {'split': name, 'rows': rows, 'positives': pos, 'negatives': rows - pos, 'positive_rate': pos / rows if rows else float('nan'), 'years': str(sorted(df['year'].unique().tolist()))}

def completed_metrics_for_fold(fold_id):
    if fold_id == 'test_2015':
        p = METRICS_DIR / 'aia_resnet18_physics_safe_fold2015_fullnatural_benchmark_fullnatural_physics_safe_metrics.json'
        if p.exists():
            return p
    p2 = METRICS_DIR / f'aia_resnet18_physics_safe_multifold_fullnatural_{fold_id}_metrics.json'
    if p2.exists():
        return p2
    return None

def train_one_fold(fold):
    fold_id = fold['fold_id']; fold_prefix = f'aia_resnet18_physics_safe_multifold_fullnatural_{fold_id}'
    progress_log = METRICS_DIR / f'{fold_prefix}_progress.log'; progress_log.write_text('')
    def log(msg):
        print(msg, flush=True)
        with open(progress_log, 'a') as fh:
            fh.write(msg + '\n')
    log(f'===== START FOLD {fold_id} =====')
    seed_everything(CFG.seed)
    train_df, val_df, test_df = build_fold_splits(fold)
    for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        df.to_csv(METRICS_DIR / f'{fold_prefix}_{name}_samples.csv', index=False)
    data_summary = [summarise_split(train_df, 'train'), summarise_split(val_df, 'val'), summarise_split(test_df, 'test')]
    log(str(data_summary))
    stats = compute_train_channel_stats(train_df, fold_prefix, progress_log)
    train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df, stats)
    pos = float(train_df[CFG.label_col].sum()); neg = float(len(train_df) - pos); pos_weight_value = neg / max(pos, 1.0)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=device)
    model = ResNet18AIA(in_channels=CFG.in_channels, dropout=CFG.dropout).to(device)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    model_path = MODELS_DIR / f'{fold_prefix}.pt'; history_path = METRICS_DIR / f'{fold_prefix}_history.csv'
    history = []; best_val_tss = -999.0; best_epoch = None
    for epoch in range(1, CFG.epochs + 1):
        model.train(); total_loss = 0.0; n = 0; t0 = time.time()
        for batch_idx, (xb, yb, _) in enumerate(train_loader, 1):
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True); logits = model(xb); loss = criterion(logits, yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0); optimizer.step()
            bs = xb.size(0); total_loss += float(loss.detach().cpu()) * bs; n += bs
            if batch_idx % 100 == 0 or batch_idx == len(train_loader):
                log(f'fold={fold_id} epoch={epoch:02d} batch={batch_idx}/{len(train_loader)} loss_running={total_loss / max(n, 1):.4f} elapsed_min={(time.time() - t0) / 60:.1f}')
        train_loss = total_loss / max(n, 1)
        val_metrics, _, _ = evaluate_split(model, val_loader, 'val', progress_log)
        val_best = val_metrics['best_tss']; val_tss = float(val_best['tss']); lr_now = float(optimizer.param_groups[0]['lr'])
        row = {'fold_id': fold_id, 'epoch': epoch, 'train_loss': train_loss, 'val_roc_auc': val_metrics['roc_auc'], 'val_pr_auc': val_metrics['pr_auc'], 'val_best_threshold': val_best['threshold'], 'val_tss': val_tss, 'val_hss': float(val_best['hss']), 'lr': lr_now}
        history.append(row); pd.DataFrame(history).to_csv(history_path, index=False)
        log(f"fold={fold_id} epoch={epoch:02d}/{CFG.epochs} COMPLETE loss={train_loss:.4f} val_auc={val_metrics['roc_auc']:.4f} val_pr={val_metrics['pr_auc']:.4f} val_thr={val_best['threshold']:.4f} val_tss={val_tss:.4f} val_hss={val_best['hss']:.4f} lr={lr_now:.2e}")
        scheduler.step(val_tss)
        if val_tss > best_val_tss:
            best_val_tss = val_tss; best_epoch = epoch
            torch.save({'model_state_dict': model.state_dict(), 'config': asdict(CFG), 'fold': fold, 'best_epoch': best_epoch, 'best_val_tss': best_val_tss, 'trainable_parameters': trainable_params, 'channel_stats': stats}, model_path)
            log(f'saved checkpoint -> {model_path}')
    log(f'Training complete fold={fold_id}. Best epoch={best_epoch}, best_val_tss={best_val_tss}')
    ckpt = torch.load(model_path, map_location=device); model.load_state_dict(ckpt['model_state_dict']); model.to(device)
    val_metrics, val_grid, val_pred = evaluate_split(model, val_loader, 'val_final', progress_log)
    test_raw, test_grid, test_pred = evaluate_split(model, test_loader, 'test_final', progress_log)
    selected_threshold = float(val_metrics['best_tss']['threshold'])
    test_at_selected = confusion_at_threshold(test_pred['y_true'].values, test_pred['y_prob'].values, selected_threshold)
    val_pred.to_csv(METRICS_DIR / f'{fold_prefix}_val_predictions.csv', index=False); test_pred.to_csv(METRICS_DIR / f'{fold_prefix}_test_predictions.csv', index=False)
    val_grid.to_csv(METRICS_DIR / f'{fold_prefix}_val_threshold_grid.csv', index=False); test_grid.to_csv(METRICS_DIR / f'{fold_prefix}_test_threshold_grid.csv', index=False)
    metrics_path = METRICS_DIR / f'{fold_prefix}_metrics.json'
    metrics = {'experiment_name': 'aia_resnet18_physics_safe_multifold_fullnatural', 'fold_id': fold_id, 'fold': fold, 'data_summary': data_summary, 'physics_safety': {'label_source': CFG.label_col, 'uses_embedded_npz_label': False, 'spatial_augmentation': False, 'random_flip': False, 'random_rotation': False, 'random_crop': False, 'normalisation': stats['normalisation'], 'normalisation_source': 'training split only per fold', 'weighted_random_sampler': False, 'threshold_protocol': 'max TSS on validation applied unchanged to test'}, 'model': {'architecture': 'ResNet18 six-channel AIA CNN', 'trainable_parameters': trainable_params, 'image_size': CFG.image_size, 'batch_size': CFG.batch_size, 'epochs': CFG.epochs, 'learning_rate': CFG.learning_rate, 'weight_decay': CFG.weight_decay, 'dropout': CFG.dropout, 'pos_weight': pos_weight_value}, 'best_epoch': int(ckpt['best_epoch']), 'selected_threshold_from_validation': selected_threshold, 'validation': val_metrics, 'test': {'roc_auc': test_raw['roc_auc'], 'pr_auc': test_raw['pr_auc'], 'brier_score': test_raw['brier_score'], 'at_0_5': test_raw['at_0_5'], 'best_tss': test_raw['best_tss'], 'at_selected_threshold': test_at_selected}}
    metrics_path.write_text(json.dumps(metrics, indent=2)); log(f'Metrics saved: {metrics_path}')
    return metrics_path

fold_metric_paths = []
for fold in FOLDS:
    fold_id = fold['fold_id']
    existing = completed_metrics_for_fold(fold_id)
    if existing is not None and not RERUN_EXISTING_FOLDS:
        print(f'Reusing existing metrics for {fold_id}: {existing.relative_to(ROOT)}')
        fold_metric_paths.append(existing)
    else:
        fold_metric_paths.append(train_one_fold(fold))
print('Fold metric paths:')
for p in fold_metric_paths:
    print('-', p.relative_to(ROOT))

def row_from_metrics(path):
    m = json.loads(Path(path).read_text()); test = m['test']['at_selected_threshold']; ds = m.get('data_summary', [])
    def split_value(split, key):
        for row in ds:
            if str(row.get('split', '')).lower() == split:
                return row.get(key, np.nan)
        return np.nan
    return {'fold_id': m.get('fold_id', Path(path).stem), 'source_file': str(Path(path).relative_to(ROOT)), 'train_rows': split_value('train', 'rows'), 'val_rows': split_value('val', 'rows'), 'test_rows': split_value('test', 'rows'), 'test_positives': split_value('test', 'positives'), 'test_negatives': split_value('test', 'negatives'), 'best_epoch': m.get('best_epoch', np.nan), 'threshold': m.get('selected_threshold_from_validation', test.get('threshold', np.nan)), 'roc_auc': m['test'].get('roc_auc', np.nan), 'pr_auc': m['test'].get('pr_auc', np.nan), 'brier_score': m['test'].get('brier_score', np.nan), 'accuracy': test.get('accuracy', np.nan), 'precision': test.get('precision', np.nan), 'recall': test.get('recall', np.nan), 'specificity': test.get('specificity', np.nan), 'f1': test.get('f1', np.nan), 'tss': test.get('tss', np.nan), 'hss': test.get('hss', np.nan), 'tp': test.get('tp', np.nan), 'tn': test.get('tn', np.nan), 'fp': test.get('fp', np.nan), 'fn': test.get('fn', np.nan), 'diagnostic_test_best_tss': m['test'].get('best_tss', {}).get('tss', np.nan)}

fold_df = pd.DataFrame([row_from_metrics(p) for p in fold_metric_paths]).sort_values('fold_id').reset_index(drop=True)
metric_cols = ['roc_auc', 'pr_auc', 'brier_score', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'tss', 'hss']
mean_std_df = pd.DataFrame([{'metric': c, 'mean': pd.to_numeric(fold_df[c], errors='coerce').mean(), 'std': pd.to_numeric(fold_df[c], errors='coerce').std(ddof=1), 'min': pd.to_numeric(fold_df[c], errors='coerce').min(), 'max': pd.to_numeric(fold_df[c], errors='coerce').max()} for c in metric_cols])

out_csv = METRICS_DIR / 'aia_resnet18_physics_safe_multifold_fullnatural_summary.csv'
out_md = METRICS_DIR / 'aia_resnet18_physics_safe_multifold_fullnatural_summary.md'
out_stats = METRICS_DIR / 'aia_resnet18_physics_safe_multifold_fullnatural_mean_std.csv'
out_json = METRICS_DIR / 'aia_resnet18_physics_safe_multifold_fullnatural_summary.json'
fold_df.round(4).to_csv(out_csv, index=False); mean_std_df.round(4).to_csv(out_stats, index=False)
md_text = '# AIA ResNet18 Physics-Safe Multi-Fold Full-Natural Summary\n\n## Fold-level official test metrics\n\n' + fold_df.round(4).to_markdown(index=False) + '\n\n## Mean ± standard deviation across folds\n\n' + mean_std_df.round(4).to_markdown(index=False) + '\n'
out_md.write_text(md_text)
out_json.write_text(json.dumps({'fold_metric_paths': [str(Path(p).relative_to(ROOT)) for p in fold_metric_paths], 'fold_level': fold_df.to_dict(orient='records'), 'mean_std': mean_std_df.to_dict(orient='records'), 'physics_safety': {'label_source': CFG.label_col, 'uses_embedded_npz_label': False, 'spatial_augmentation': False, 'random_flip': False, 'random_rotation': False, 'random_crop': False, 'threshold_protocol': 'validation-selected max-TSS per fold'}}, indent=2))
display(fold_df.round(4)); display(mean_std_df.round(4))
print('Saved:')
for p in [out_csv, out_md, out_stats, out_json]:
    print('-', p.relative_to(ROOT))

import matplotlib.pyplot as plt
for metric in ['tss', 'recall', 'specificity', 'precision', 'roc_auc', 'pr_auc', 'hss']:
    labels = fold_df['fold_id'].tolist(); vals = pd.to_numeric(fold_df[metric], errors='coerce')
    plt.figure(figsize=(8, 4.5)); plt.bar(labels, vals); plt.ylabel(metric.upper().replace('_', '-'))
    plt.title(f'ResNet18 full-natural multi-fold: {metric.upper().replace("_", "-")}')
    plt.tight_layout(); fig_path = FIG_DIR / f'aia_resnet18_multifold_fullnatural_{metric}.png'
    plt.savefig(fig_path, dpi=200); plt.show(); print('Saved', fig_path.relative_to(ROOT))

mean_tss = mean_std_df.loc[mean_std_df['metric'] == 'tss', 'mean'].iloc[0]; std_tss = mean_std_df.loc[mean_std_df['metric'] == 'tss', 'std'].iloc[0]
mean_recall = mean_std_df.loc[mean_std_df['metric'] == 'recall', 'mean'].iloc[0]; std_recall = mean_std_df.loc[mean_std_df['metric'] == 'recall', 'std'].iloc[0]
mean_auc = mean_std_df.loc[mean_std_df['metric'] == 'roc_auc', 'mean'].iloc[0]; std_auc = mean_std_df.loc[mean_std_df['metric'] == 'roc_auc', 'std'].iloc[0]
mean_precision = mean_std_df.loc[mean_std_df['metric'] == 'precision', 'mean'].iloc[0]
interpretation_lines = [
    '# Interpretation: ResNet18 Physics-Safe Multi-Fold Full-Natural Benchmark',
    '',
    'The multi-fold ResNet18 benchmark evaluates image-only AIA flare forecasting across the usable chronological folds `test_2013`, `test_2014`, and `test_2015`. Each fold uses train-only robust channel normalisation, no spatial augmentation, no embedded NPZ labels, and validation-selected max-TSS thresholding applied unchanged to the test year.',
    '',
    f'Across folds, ResNet18 achieved mean **TSS={mean_tss:.4f} ± {std_tss:.4f}**, mean **Recall={mean_recall:.4f} ± {std_recall:.4f}**, and mean **ROC-AUC={mean_auc:.4f} ± {std_auc:.4f}**. Precision remains low, with mean precision around **{mean_precision:.4f}**, reflecting severe natural class imbalance and the operational preference for high event sensitivity.',
    '',
    'These results are more defensible than a single test year because they test year-wise stability under chronological distribution shift. They should be interpreted as a high-recall, false-alarm-heavy image-only benchmark for later comparison against AIA+SHARP fusion models.',
    ''
]
interpretation = '\n'.join(interpretation_lines)
interp_path = METRICS_DIR / 'aia_resnet18_physics_safe_multifold_fullnatural_interpretation.md'
interp_path.write_text(interpretation); display(Markdown(interpretation)); print('Saved', interp_path.relative_to(ROOT))


## Backup and commit reminder

After execution, back up `results/metrics/aia_resnet18_physics_safe_multifold_fullnatural*`, `results/figures/aia_resnet18_multifold_fullnatural_*.png`, this training notebook, the executed notebook, and the run log to `gs://suryabench-sharp-pipeline-bamidele/aia_cnn_baselines/resnet18_multifold_fullnatural_physics_safe/`. Then commit those notebook, metric, and figure files to GitHub.
